# Exploratory Data Analysis — Univariate EDA

**Contributors:** Jeff [Nachname], [Teammate 1], [Teammate 2]

**Course:** Data Analytics & Applications, TH Nürnberg (Georg Simon Ohm)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scipy.stats as stats

---
# 1) Autonomous Vehicle Survey of Bicyclists and Pedestrians in Pittsburgh

## 1.1 Load the survey data for 2019

In [ ]:
# Load the combined 2019 survey data prepared in the previous lab (adjust filepath if needed)
df_av = pd.read_csv("../data/av_survey_data/avsurvey2019data.csv", parse_dates=["Start Date", "End Date"])
print(f"Shape: {df_av.shape}")
df_av.head()

In [ ]:
df_av.info()

## 1.2 Bar chart of the top 5 variables with the most missing values

In [ ]:
# Count missing values per column and select the top 5
missing = df_av.isnull().sum().sort_values(ascending=False).head(5)

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.bar(missing.index, missing.values, color='#4f6272', edgecolor='black')

# Add value labels and percentage on top of each bar
total_rows = len(df_av)
for bar, val in zip(bars, missing.values):
    pct = val / total_rows * 100
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 2,
            f"{val} ({pct:.1f}%)", ha='center', va='bottom', fontsize=10)

ax.set_title('Top 5 Variables with the Most Missing Values (2019 AV Survey)', fontsize=13)
ax.set_ylabel('Number of Missing Values')
ax.set_xlabel('Variable')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.show()

**Interpretation:** The bar chart above shows the five variables with the highest number of missing values. The absolute counts and percentages are displayed on each bar for easy comparison. Variables with many missing values may indicate conditional questions (e.g. follow-up questions that only apply to a subset of respondents) or questions that respondents chose to skip.

## 1.3 Analysis of a nominal and a Likert-type variable

We select:
- **Nominal variable:** `Gender` — a categorical variable without inherent order.
- **Likert-type (ordinal) variable:** `SafetyAV` — "How safe would you feel using Pittsburgh's streets with autonomous (self-driving) vehicles?" rated on a scale from 1 (very unsafe) to 5 (very safe).

*Note:* Adjust the variable names below if your 2019 dataset uses different column names.

### 1.3.1 Nominal variable: `Gender`

In [ ]:
# Frequency table for Gender (including missing values)
gender_counts = df_av['Gender'].value_counts(dropna=False)
gender_pct = df_av['Gender'].value_counts(normalize=True, dropna=False).mul(100).round(1)

gender_table = pd.DataFrame({'Count': gender_counts, 'Percentage (%)': gender_pct})
display(gender_table)

In [ ]:
# Bar chart for Gender — ordered by frequency (appropriate for nominal data)
gender_sorted = df_av['Gender'].value_counts(dropna=False)

fig, ax = plt.subplots(figsize=(8, 5))
bars = gender_sorted.plot(kind='bar', color='#4f6272', edgecolor='black', ax=ax)

# Add count labels
for container in ax.containers:
    ax.bar_label(container)

ax.set_title('Distribution of Gender (2019 AV Survey)', fontsize=13)
ax.set_xlabel('Gender')
ax.set_ylabel('Number of Respondents')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

**Interpretation — Gender:**
- **Dominant category:** Inspect the frequency table above to identify the most frequent response.
- **Balance:** The categories are likely imbalanced; the survey may over-represent certain gender groups, which is common in self-selected survey samples of cyclists and pedestrians.
- Since Gender is a **nominal** variable (no inherent order), the bars are sorted by frequency.

### 1.3.2 Likert-type (ordinal) variable: `SafetyAV`

In [ ]:
# Frequency table for SafetyAV (including missing values)
safety_counts = df_av['SafetyAV'].value_counts(dropna=False).sort_index()
safety_pct = df_av['SafetyAV'].value_counts(normalize=True, dropna=False).sort_index().mul(100).round(1)

safety_table = pd.DataFrame({'Count': safety_counts, 'Percentage (%)': safety_pct})
display(safety_table)

In [ ]:
# Bar chart for SafetyAV — sorted by scale value (preserving ordinal order)
safety_sorted = df_av['SafetyAV'].value_counts(dropna=False).sort_index(ascending=True)

fig, ax = plt.subplots(figsize=(10, 5))
safety_sorted.plot(kind='bar', color='#a8eca2', edgecolor='black', ax=ax)

for container in ax.containers:
    ax.bar_label(container)

ax.set_title('SafetyAV — Perceived Safety with Autonomous Vehicles (1: very unsafe, 5: very safe)', fontsize=12)
ax.set_xlabel('Safety Rating')
ax.set_ylabel('Number of Respondents')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

**Interpretation — SafetyAV:**
- **Dominant category:** Identify the rating with the highest bar / count above.
- **Balance:** If responses are concentrated at the lower end of the scale, the distribution is skewed towards feeling unsafe — categories are not balanced.
- The bars are ordered from 1 to 5 (plus NaN), preserving the **ordinal nature** of the Likert scale.

### Why is it inappropriate to calculate a mean for the ordinal variable?

A Likert scale is **ordinal**, meaning the categories have a meaningful order, but the **distances between categories are not guaranteed to be equal**. The difference between "1 — very unsafe" and "2 — somewhat unsafe" is not necessarily the same as between "4 — somewhat safe" and "5 — very safe." 

Calculating a mean assumes equal intervals (i.e., an interval or ratio scale), which is not justified for ordinal data. Appropriate summary measures for ordinal data are the **median** and the **mode**. Some researchers still report means for Likert-type data for practical comparisons, but this should be done with caution and explicitly acknowledged.

---
# 2) Degradation Measurement of Robot Arm Position Accuracy

## 2.1 Select one trial for an experimental condition

In [ ]:
# Load the combined parquet dataset prepared in the previous lab (adjust filepath if needed)
df_robot_all = pd.read_parquet("../data/robot_position_accuracy/ur5testresult_all.parquet")
print(f"Full dataset shape: {df_robot_all.shape}")

In [ ]:
# Select one experimental condition: load=1.6 lbs, full speed, no cold start, trial 1
df_robot = df_robot_all[
    (df_robot_all['load'] == 1.6) &
    (df_robot_all['speed'] == 'full') &
    (df_robot_all['cold_start'] == False) &
    (df_robot_all['test_id'] == 1)
].copy()

print(f"Filtered dataset shape: {df_robot.shape}")
df_robot.head()

**Chosen condition:** Load = 1.6 lbs, full speed, no cold start, trial 1.

We chose this condition as a representative baseline run (low load, standard speed, warmed-up robot).

## 2.2 Compute the difference between actual and target current in J2 and J3

The difference `actual - target` tells us how far the robot's actual joint current deviated from the planned target current during operation. We inspect this for joints J2 and J3.

*Note:* Adjust column names below to match your dataset (e.g. `ACTUAL_CURRENT_J2`, `TARGET_CURRENT_J2`, etc.).

In [ ]:
# Identify the correct column names for target and actual current in J2 and J3
# Uncomment the line below to inspect available column names if needed:
# [col for col in df_robot.columns if 'CURRENT' in col.upper() or 'current' in col.lower()]

In [ ]:
# Compute differences — adjust column names to your dataset!
# Common naming patterns:
#   Parquet with cleaned names: 'actual_current_j2', 'target_current_j2'
#   Raw style: 'ACTUAL_CURRENT (J2)', 'TARGET_CURRENT (J2)'

# Try to find columns automatically
current_cols = [col for col in df_robot.columns if 'current' in col.lower()]
print("Available current columns:")
print(current_cols)

In [ ]:
# --- Adjust these column names to match your parquet file! ---
# Example for snake_case columns:
actual_j2_col = 'actual_current_j2'   # <- adjust if needed
target_j2_col = 'target_current_j2'   # <- adjust if needed
actual_j3_col = 'actual_current_j3'   # <- adjust if needed
target_j3_col = 'target_current_j3'   # <- adjust if needed

df_robot['diff_current_j2'] = df_robot[actual_j2_col] - df_robot[target_j2_col]
df_robot['diff_current_j3'] = df_robot[actual_j3_col] - df_robot[target_j3_col]

df_robot[['diff_current_j2', 'diff_current_j3']].describe()

### Comparison of descriptive statistics

In [ ]:
# Side-by-side descriptive statistics including skewness and kurtosis
desc = df_robot[['diff_current_j2', 'diff_current_j3']].describe()
desc.loc['skewness'] = df_robot[['diff_current_j2', 'diff_current_j3']].skew()
desc.loc['kurtosis'] = df_robot[['diff_current_j2', 'diff_current_j3']].kurtosis()
desc.loc['median'] = df_robot[['diff_current_j2', 'diff_current_j3']].median()
display(desc)

### Histograms with normal fit

In [ ]:
diff_cols = ['diff_current_j2', 'diff_current_j3']
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for i, col in enumerate(diff_cols):
    ax = axes[i]
    data = df_robot[col].dropna()
    
    # Histogram (density for comparison with normal curve)
    ax.hist(data, bins=40, color='#4f6272', edgecolor='black', density=True, alpha=0.7)
    
    # Fitted normal distribution
    mu, std = stats.norm.fit(data)
    x = np.linspace(data.min(), data.max(), 200)
    ax.plot(x, stats.norm.pdf(x, mu, std), 'b-', linewidth=2, label=f'Normal fit\n(μ={mu:.4f}, σ={std:.4f})')
    
    ax.set_title(f'Distribution of {col}', fontsize=12)
    ax.set_xlabel('Current difference (A)')
    ax.set_ylabel('Density')
    ax.legend()

plt.tight_layout()
plt.show()

### Boxplots

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for i, col in enumerate(diff_cols):
    ax = axes[i]
    df_robot[[col]].dropna().plot(
        kind='box', ax=ax, vert=True,
        showmeans=True,
        meanprops=dict(marker='x', markeredgecolor='blue', markerfacecolor='blue', markersize=10),
        # Default whisker length is 1.5 * IQR; we use 2.5 * IQR as specified in the task
        whis=2.5
    )
    ax.set_title(f'Boxplot of {col} (whis=2.5)', fontsize=12)
    ax.set_ylabel('Current difference (A)')

plt.tight_layout()
plt.show()

### Outlier detection (2.5 × IQR)

In [ ]:
for col in diff_cols:
    data = df_robot[col].dropna()
    Q1 = data.quantile(0.25)
    Q3 = data.quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 2.5 * IQR
    upper_bound = Q3 + 2.5 * IQR
    
    outliers = data[(data < lower_bound) | (data > upper_bound)]
    
    print(f"--- {col} ---")
    print(f"  Q1 = {Q1:.6f}, Q3 = {Q3:.6f}, IQR = {IQR:.6f}")
    print(f"  Lower bound: {lower_bound:.6f}")
    print(f"  Upper bound: {upper_bound:.6f}")
    print(f"  Number of outliers (2.5 × IQR): {len(outliers)}")
    print()

### Q-Q Plots

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for i, col in enumerate(diff_cols):
    ax = axes[i]
    stats.probplot(df_robot[col].dropna(), dist='norm', plot=ax)
    ax.set_title(f'Q-Q Plot (Normal) — {col}', fontsize=12)

plt.tight_layout()
plt.show()

## 2.3 Answers to the questions

### Are there outliers based on a distance of at least 2.5 × IQR from the upper/lower quartile?

See the outlier counts printed above. Using the stricter 2.5 × IQR criterion (instead of the standard 1.5 × IQR), outliers are those data points that fall outside the range [Q1 − 2.5·IQR, Q3 + 2.5·IQR]. The boxplots with `whis=2.5` visualize this directly — any points plotted beyond the whiskers are flagged as outliers.

→ *Fill in:* Report the number of outliers for J2 and J3 from the output above.

### Is the distribution symmetric or skewed?

We can assess symmetry from:
- The **skewness** values in the descriptive statistics table (values close to 0 indicate symmetry; positive values indicate right-skew, negative values indicate left-skew).
- The **histograms** — a visually symmetric bell shape vs. a tail extending to one side.
- The **boxplots** — if the median line is centered within the box and the whiskers are roughly equal in length, the distribution is approximately symmetric.

→ *Fill in:* Describe whether J2 and J3 are symmetric or skewed based on the plots and statistics above.

### Is it unimodal?

A unimodal distribution has a single peak. We can assess this from the **histograms** — if there is clearly one dominant peak, the distribution is unimodal. Multiple peaks would indicate bimodal or multimodal distributions, which could occur if different operating states are mixed.

→ *Fill in:* Describe modality based on the histogram shapes.

### Is the normality assumption plausible? Why?

We assess normality through multiple perspectives:
1. **Histograms with normal fit:** Does the fitted normal curve approximate the observed data well?
2. **Q-Q plots:** If the data points closely follow the diagonal line, the normal distribution is a plausible model. Systematic deviations (S-shapes, heavy tails) suggest departures from normality.
3. **Skewness and kurtosis:** For a normal distribution, skewness ≈ 0 and excess kurtosis ≈ 0. Large deviations indicate non-normality.

→ *Fill in:* Summarize your assessment based on the Q-Q plots, histograms, and descriptive statistics.